In [1]:
import time
import re
import pandas as pd
import chromedriver_autoinstaller

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# =========================
# 0. 기본 설정 & 파일 경로
# =========================

################################ 중간에 ## 안에 각자 할당된 숫자 입력해주세요.(INPUT/OUTPUT 둘 다 해당)

INPUT_XLSX_PATH = "yeoshinticket_event_urls_part_1_of_5.xlsx"
OUTPUT_CSV_PATH = "yeoshinticket_detail_procedure_final_1.csv"

MAX_RETRY = 3          # URL당 재시도 횟수
SLEEP_BETWEEN = 0.7    # 요청 사이 딜레이(속도/안정성 타협값)


# =========================
# 1. 크롬 드라이버 설정
# =========================

def get_driver():
    chromedriver_autoinstaller.install()
    chrome_options = Options()
#    chrome_options.add_argument("--headless=new")  # 디버깅할 땐 주석 처리
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--window-size=1920,1080")
    driver = webdriver.Chrome(options=chrome_options)
    driver.implicitly_wait(5)
    return driver


# =========================
# 2. 유틸 함수들
# =========================

def safe_get_text(driver, selector):
    try:
        return driver.find_element(By.CSS_SELECTOR, selector).text.strip()
    except Exception:
        return ""


def safe_get_attr(driver, selector, attr):
    try:
        el = driver.find_element(By.CSS_SELECTOR, selector)
        val = el.get_attribute(attr)
        return val.strip() if val else ""
    except Exception:
        return ""


def safe_get_section_texts_joined(driver, selector, sep=" / "):
    try:
        elements = driver.find_elements(By.CSS_SELECTOR, selector + " *")
        texts = [e.text.strip() for e in elements if e.text.strip()]
        return sep.join(texts)
    except Exception:
        return ""


def get_hospital_address(driver):
    """병원 주소 추출 (주소복사 텍스트 제거)"""
    css_addr = (
        "#ct-view > div.relative.w-full > div.relative.bg-white > "
        "div.flex.flex-col.mt-\\[32px\\].justify-center.w-full.gap-\\[32px\\] > "
        "article > section.flex.w-full.px-\\[16px\\] > div > div > div"
    )
    try:
        el = driver.find_element(By.CSS_SELECTOR, css_addr)
        text = el.text.strip()
    except Exception:
        return ""

    if "주소복사" in text:
        text = text.replace("주소복사", "").strip()

    return text


def get_weekly_opening_hours(driver):
    """영업시간 토글 열고 일주일 영업시간 텍스트 가져오기"""
    css_hours_button = "#radix-\\:Rl3sckum\\:"
    css_hours_panel = "#radix-\\:R1l3sckum\\:"

    try:
        btn = driver.find_element(By.CSS_SELECTOR, css_hours_button)
        driver.execute_script("arguments[0].click();", btn)
        time.sleep(0.4)
    except Exception:
        pass

    try:
        panel = driver.find_element(By.CSS_SELECTOR, css_hours_panel)
        raw_text = panel.text
        lines = [ln.strip() for ln in raw_text.splitlines() if ln.strip()]
        return " / ".join(lines)
    except Exception:
        return ""


def get_hospital_contact(driver):
    """
    병원 연락처 추출 (class 기반, nth-child 안 씀)
    1) 페이지 내 text-gray700... 블록 전체 탐색
    2) 각 블록의 모든 줄 split
    3) 전화번호 패턴(숫자+하이픈) 있는 줄 우선 선택
    """
    containers = driver.find_elements(
        By.CSS_SELECTOR,
        "div.text-gray700.font-normal.text-\\[12px\\].leading-\\[18px\\]"
    )

    texts = []
    for c in containers:
        try:
            for ln in c.text.splitlines():
                ln = ln.strip()
                if ln:
                    texts.append(ln)
        except Exception:
            continue

    if not texts:
        return ""

    phone_pattern = re.compile(r"\d{2,4}-\d{3,4}-\d{4}")

    # 1순위: 전화번호 패턴이 들어있는 줄
    for t in texts:
        if phone_pattern.search(t):
            return t

    # 2순위: 숫자가 가장 많이 들어있는 줄
    texts_sorted = sorted(texts, key=lambda x: sum(ch.isdigit() for ch in x), reverse=True)
    best = texts_sorted[0]
    if sum(ch.isdigit() for ch in best) < 4:
        return ""
    return best


# =========================
# 3. 메인 크롤링 로직 (전체 URL 대상)
# =========================

def crawl_yeoshin_event_details():
    # 1) 엑셀 전체 로드
    df = pd.read_excel(INPUT_XLSX_PATH)

    required_cols = ["대분류", "중분류", "소분류", "event_url"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"엑셀에 '{col}' 컬럼이 없습니다.")

    total = len(df)
    print(f"👉 전체 {total}개 event_url 크롤링을 시작합니다.\n")

    driver = get_driver()
    wait = WebDriverWait(driver, 10)
    results = []

    try:
        for idx, row in df.iterrows():
            depth1 = row["대분류"]
            depth2 = row["중분류"]
            depth3 = row["소분류"]
            url = row["event_url"]

            print(f"[{idx+1}/{total}] {url}")

            if not isinstance(url, str) or not url.startswith("http"):
                print("  ⚠ 잘못된 URL 형식, 스킵")
                continue

            success = False

            for attempt in range(1, MAX_RETRY + 1):
                try:
                    driver.get(url)

                    # 시술명 등장까지 대기(페이지 로딩 기준)
                    try:
                        wait.until(
                            EC.presence_of_element_located(
                                (
                                    By.CSS_SELECTOR,
                                    "#ct-view > div.relative.w-full > div.relative.bg-white > "
                                    "div.px-\\[16px\\].relative.bg-white.w-full > "
                                    "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
                                    "section.flex.flex-col.gap-\\[4px\\] > h1"
                                )
                            )
                        )
                    except Exception:
                        pass

                    # ---------- 시술 영역 ----------
                    css_title = (
                        "#ct-view > div.relative.w-full > div.relative.bg-white > "
                        "div.px-\\[16px\\].relative.bg-white.w-full > "
                        "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
                        "section.flex.flex-col.gap-\\[4px\\] > h1"
                    )
                    css_rep_image = (
                        "#ct-view > div.relative.w-full > div.relative.overflow-hidden > div > img"
                    )
                    css_hashtags = (
                        "#ct-view > div.relative.w-full > div.relative.bg-white > "
                        "div.px-\\[16px\\].relative.bg-white.w-full > "
                        "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
                        "section.flex.flex-col.gap-\\[4px\\] > div"
                    )
                    css_rating = (
                        "#ct-view > div.relative.w-full > div.relative.bg-white > "
                        "div.px-\\[16px\\].relative.bg-white.w-full > "
                        "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
                        "section.flex.flex-col.gap-\\[4px\\] > button > div > span"
                    )
                    css_review_cnt = (
                        "#ct-view > div.relative.w-full > div.relative.bg-white > "
                        "div.px-\\[16px\\].relative.bg-white.w-full > "
                        "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
                        "section.flex.flex-col.gap-\\[4px\\] > button > span"
                    )
                    css_price_original = (
                        "#ct-view > div.relative.w-full > div.relative.bg-white > "
                        "div.px-\\[16px\\].relative.bg-white.w-full > "
                        "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
                        "section.flex.items-end.justify-between.w-full.mt-\\[8px\\] > "
                        "div > div:nth-child(1) > div"
                    )
                    css_vat = (
                        "#ct-view > div.relative.w-full > div.relative.bg-white > "
                        "div.px-\\[16px\\].relative.bg-white.w-full > "
                        "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
                        "section.flex.items-end.justify-between.w-full.mt-\\[8px\\] > "
                        "div > div.flex.items-end.gap-\\[4px\\] > "
                        "div.text-\\[12px\\].mb-\\[2px\\].font-normal.leading-\\[18px\\].text-gray500"
                    )
                    css_treatment_info = (
                        "#ct-view > div.relative.w-full > div.relative.bg-white > "
                        "section.flex.flex-col.px-\\[16px\\].py-\\[24px\\] > div:nth-child(1)"
                    )

                    title = safe_get_text(driver, css_title)
                    rep_image = safe_get_attr(driver, css_rep_image, "src")
                    hashtags = safe_get_text(driver, css_hashtags)
                    rating = safe_get_text(driver, css_rating)
                    review_cnt = safe_get_text(driver, css_review_cnt)
                    price_original = safe_get_text(driver, css_price_original)
                    vat_info = safe_get_text(driver, css_vat)
                    treatment_info = safe_get_text(driver, css_treatment_info)

                    # ---------- 병원 영역 ----------
                    css_hospital_name = (
                        "#ct-view > div.relative.w-full > div.relative.bg-white > "
                        "div.flex.flex-col.mt-\\[32px\\].justify-center.w-full.gap-\\[32px\\] > "
                        "article > div > div > h2"
                    )
                    css_hospital_info_section = (
                        "#ct-view > div.relative.w-full > div.relative.bg-white > "
                        "div.flex.flex-col.mt-\\[32px\\].justify-center.w-full.gap-\\[32px\\] > "
                        "article > section.flex.flex-wrap.gap-\\[4px\\].px-\\[16px\\]"
                    )
                    css_anesthesia_info = (
                        "#ct-view > div.relative.w-full > div.relative.bg-white > "
                        "div.flex.flex-col.mt-\\[32px\\].justify-center.w-full.gap-\\[32px\\] > "
                        "article > section.flex.flex-col.gap-\\[16px\\].px-\\[16px\\] > div > div"
                    )

                    hospital_name = safe_get_text(driver, css_hospital_name)
                    hospital_info = safe_get_section_texts_joined(
                        driver, css_hospital_info_section
                    )
                    hospital_addr = get_hospital_address(driver)
                    anesthesia_info = safe_get_text(driver, css_anesthesia_info)
                    opening_hours = get_weekly_opening_hours(driver)
                    hospital_contact = get_hospital_contact(driver)

                    row_data = {
                        "대분류": depth1,
                        "중분류": depth2,
                        "소분류": depth3,
                        "event_url": url,
                        "시술명": title,
                        "대표이미지": rep_image,
                        "시술_해시태그": hashtags,
                        "평점": rating,
                        "후기_갯수": review_cnt,
                        "원가": price_original,
                        "VAT_정보": vat_info,
                        "시술_정보": treatment_info,
                        "병원명": hospital_name,
                        "병원_정보": hospital_info,
                        "병원_주소": hospital_addr,
                        "마취정보": anesthesia_info,
                        "영업시간": opening_hours,
                        "병원_연락처(안심번호)": hospital_contact,
                    }

                    results.append(row_data)
                    success = True
                    time.sleep(SLEEP_BETWEEN)
                    break

                except Exception as e:
                    print(f"  ❌ 시도 {attempt}/{MAX_RETRY} 실패: {e}")
                    time.sleep(1.0)

            if not success:
                print("  ⚠ 최종 실패: 이 URL은 수집하지 못했습니다.")

    finally:
        driver.quit()

    if results:
        df_out = pd.DataFrame(results)
        df_out.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")
        print(f"\n✅ 크롤링 완료! 총 {len(df_out)}건 저장")
        print(f"📁 저장 파일: {OUTPUT_CSV_PATH}")
    else:
        print("\n⚠ 수집된 결과가 없습니다. 엑셀/셀렉터를 다시 확인해 주세요.")


if __name__ == "__main__":
    crawl_yeoshin_event_details()

👉 전체 925개 event_url 크롤링을 시작합니다.

[1/925] https://www.yeoshin.co.kr/event/mobile/16693
[2/925] https://www.yeoshin.co.kr/event/mobile/25183
[3/925] https://www.yeoshin.co.kr/event/mobile/27687
[4/925] https://www.yeoshin.co.kr/event/mobile/5421
[5/925] https://www.yeoshin.co.kr/event/mobile/26604
[6/925] https://www.yeoshin.co.kr/event/mobile/20567
[7/925] https://www.yeoshin.co.kr/event/mobile/18540
[8/925] https://www.yeoshin.co.kr/event/mobile/21364
[9/925] https://www.yeoshin.co.kr/event/mobile/23106
[10/925] https://www.yeoshin.co.kr/event/mobile/26166
[11/925] https://www.yeoshin.co.kr/event/mobile/25842
[12/925] https://www.yeoshin.co.kr/event/mobile/21917
[13/925] https://www.yeoshin.co.kr/event/mobile/27778
[14/925] https://www.yeoshin.co.kr/event/mobile/27652
[15/925] https://www.yeoshin.co.kr/event/mobile/27353
[16/925] https://www.yeoshin.co.kr/event/mobile/26176
[17/925] https://www.yeoshin.co.kr/event/mobile/25153
[18/925] https://www.yeoshin.co.kr/event/mobile/17376
[19/9

In [1]:
import time
import re
import pandas as pd
import chromedriver_autoinstaller

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# =========================
# 0. 기본 설정 & 파일 경로
# =========================

# 🔹 너가 가진 "원가 결측 URL만 모아둔" 엑셀 파일명으로 수정해서 쓰면 됨
INPUT_XLSX_PATH = "yeoshinticket_missing_original_price_urls.xlsx"

# 🔹 재수집 결과 저장 파일
OUTPUT_CSV_PATH = "yeoshinticket_original_price_recrawl.csv"
OUTPUT_XLSX_PATH = "yeoshinticket_original_price_recrawl.xlsx"

MAX_RETRY = 3          # URL당 재시도 횟수
SLEEP_BETWEEN = 0.7    # 요청 사이 딜레이


# =========================
# 1. 크롬 드라이버 설정
# =========================

def get_driver():
    chromedriver_autoinstaller.install()
    chrome_options = Options()
    # 디버깅할 땐 창이 보이도록 headless 주석 처리
    # chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--window-size=1920,1080")
    driver = webdriver.Chrome(options=chrome_options)
    driver.implicitly_wait(5)
    return driver


# =========================
# 2. 유틸 함수
# =========================

def safe_get_text(driver, selector):
    try:
        return driver.find_element(By.CSS_SELECTOR, selector).text.strip()
    except Exception:
        return ""


def get_treatment_title(driver):
    """시술명 추출 (없어도 큰 문제는 없지만 검증용으로 같이 가져옴)"""
    css_title = (
        "#ct-view > div.relative.w-full > div.relative.bg-white > "
        "div.px-\\[16px\\].relative.bg-white.w-full > "
        "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
        "section.flex.flex-col.gap-\\[4px\\] > h1"
    )
    return safe_get_text(driver, css_title)


def get_original_price(driver):
    """
    '원가'(정가)를 최대한 유연하게 가져오는 함수.

    1) 기존에 쓰던 price 영역 셀렉터 우선 시도
    2) 실패하면, 가격 영역(section.flex.items-end.justify-between...) 안에서
       '원'이 들어 있고 숫자가 포함된 텍스트 중 가장 그럴듯한 것을 선택
    """

    # 1) 이전 코드에서 사용하던 셀렉터 (DOM 구조가 같은 경우)
    css_price_original = (
        "#ct-view > div.relative.w-full > div.relative.bg-white > "
        "div.px-\\[16px\\].relative.bg-white.w-full > "
        "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
        "section.flex.items-end.justify-between.w-full.mt-\\[8px\\] > "
        "div > div:nth-child(1)"
    )

    text = safe_get_text(driver, css_price_original)
    # 숫자+원 이 포함돼 있으면 그대로 사용
    if re.search(r"\d", text) and "원" in text:
        return text

    # 2) 유연한 방식: 가격 영역 전체에서 숫자+원 이 들어간 텍스트 탐색
    try:
        price_section = driver.find_element(
            By.CSS_SELECTOR,
            "#ct-view > div.relative.w-full > div.relative.bg-white > "
            "div.px-\\[16px\\].relative.bg-white.w-full > "
            "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
            "section.flex.items-end.justify-between.w-full.mt-\\[8px\\]"
        )
    except Exception:
        return ""

    candidates = []
    try:
        # 가격 섹션 내 모든 요소 텍스트 수집
        elements = price_section.find_elements(By.XPATH, ".//*")
        for el in elements:
            t = el.text.strip()
            if not t:
                continue
            # 숫자 + '원' 이 포함된 경우만 후보로
            if "원" in t and re.search(r"\d", t):
                candidates.append(t)
    except Exception:
        pass

    if not candidates:
        return ""

    # 가장 긴 텍스트(보통 "정가 110,000원" / "77,100원" 같은 값)를 선택
    candidates_sorted = sorted(candidates, key=len, reverse=True)
    return candidates_sorted[0]


# =========================
# 3. 메인: 원가 재크롤링
# =========================

def recrawl_original_price_only():
    # 1) 엑셀 로드
    df = pd.read_excel(INPUT_XLSX_PATH)

    if "event_url" not in df.columns:
        raise ValueError("엑셀 파일에 'event_url' 컬럼이 없습니다.")

    # 혹시 중복 URL이 있다면 제거
    df = df.drop_duplicates(subset=["event_url"]).reset_index(drop=True)

    total = len(df)
    print(f"👉 원가 재수집 대상 URL: {total}개\n")

    driver = get_driver()
    wait = WebDriverWait(driver, 10)
    results = []

    try:
        for idx, row in df.iterrows():
            url = row["event_url"]
            print(f"[{idx+1}/{total}] {url}")

            if not isinstance(url, str) or not url.startswith("http"):
                print("  ⚠ 잘못된 URL 형식, 스킵")
                continue

            success = False
            new_price = ""
            title = ""

            for attempt in range(1, MAX_RETRY + 1):
                try:
                    driver.get(url)

                    # 시술명 나타날 때까지 대략 대기 (페이지 로딩 기준)
                    try:
                        wait.until(
                            EC.presence_of_element_located(
                                (
                                    By.CSS_SELECTOR,
                                    "#ct-view > div.relative.w-full > div.relative.bg-white > "
                                    "div.px-\\[16px\\].relative.bg-white.w-full > "
                                    "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
                                    "section.flex.flex-col.gap-\\[4px\\] > h1"
                                )
                            )
                        )
                    except Exception:
                        pass

                    title = get_treatment_title(driver)
                    new_price = get_original_price(driver)

                    print(f"   → 시술명: {title}")
                    print(f"   → 새 원가: {new_price}")

                    success = True
                    time.sleep(SLEEP_BETWEEN)
                    break

                except Exception as e:
                    print(f"  ❌ 시도 {attempt}/{MAX_RETRY} 실패: {e}")
                    time.sleep(1.0)

            if not success:
                print("  ⚠ 최종 실패: 이 URL은 원가를 가져오지 못했습니다.")

            results.append(
                {
                    "event_url": url,
                    "시술명": title,
                    "새_원가": new_price,  # 나중에 merge 할 때 이 컬럼 사용
                }
            )

    finally:
        driver.quit()

    # 4) 저장
    if results:
        df_out = pd.DataFrame(results)
        df_out.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")
        df_out.to_excel(OUTPUT_XLSX_PATH, index=False)
        print("\n✅ 재수집 완료!")
        print(f"📁 CSV : {OUTPUT_CSV_PATH}")
        print(f"📁 XLSX: {OUTPUT_XLSX_PATH}")
    else:
        print("\n⚠ 수집된 결과가 없습니다.")


if __name__ == "__main__":
    recrawl_original_price_only()


👉 원가 재수집 대상 URL: 319개

[1/319] https://www.yeoshin.co.kr/event/mobile/25842
   → 시술명: 대표원장 1:1 디자인 리프팅 울쎄라
   → 새 원가: 278,000원
VAT 포함
[2/319] https://www.yeoshin.co.kr/event/mobile/26740
   → 시술명: 대표원장 1:1 맞춤 시술 울쎄라 400샷
   → 새 원가: 1,183,000원
VAT 포함
[3/319] https://www.yeoshin.co.kr/event/mobile/24105
   → 시술명: 프리미엄 울쎄라 패키지
   → 새 원가: 1,302,000원
VAT 포함
[4/319] https://www.yeoshin.co.kr/event/mobile/25845
   → 시술명: [대표원장]인모드는 길게 롱모드 풀페이스
   → 새 원가: 213,000원
VAT 포함
[5/319] https://www.yeoshin.co.kr/event/mobile/17247
   → 시술명: V라인 인모드 리프팅
   → 새 원가: 246,000원
VAT 포함
[6/319] https://www.yeoshin.co.kr/event/mobile/9558
   → 시술명: 인모드 여자피부과전문의
   → 새 원가: 153,000원
VAT 포함
[7/319] https://www.yeoshin.co.kr/event/mobile/17246
   → 시술명: 탄력 UP 슈링크 유니버스
   → 새 원가: 158,000원
VAT 포함
[8/319] https://www.yeoshin.co.kr/event/mobile/16042
   → 시술명: 슈링크 유니버스 300샷
   → 새 원가: 92,000원
VAT 포함
[9/319] https://www.yeoshin.co.kr/event/mobile/24652
   → 시술명: 대표원장 상담 시술 슈링크유니버스
   → 새 원가: 57,900원
VAT 포함
[10/319] htt